# 07 — Final Model Evaluation

**QuintaVision · P03 Hotel-booking cancellations**

This notebook performs the **single final held-out test evaluation** after model selection is carried out using training-only cross-validation. It compares all five required model families, plots ROC and precision–recall curves, shows confusion matrices at the default 0.5 threshold, and compares Random Forest feature importance with Logistic Regression coefficients.

> The cost-sensitive threshold decision is intentionally left to Notebook 08.

In [ ]:
from pathlib import Path
import re
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

REPO_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.project_utils import (
    RANDOM_STATE,
    TARGET,
    BOOKING_TIME_EXCLUSIONS,
    prepare_model_data,
    stratified_train_test_split,
    make_cv,
    make_random_forest,
    build_preprocessors,
)

CV = make_cv()

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Random state:", RANDOM_STATE)
print("Figures directory:", FIGURES_DIR)


## 1. Load the Canonical Cleaned Dataset

Notebook 07 consumes the cleaned artifact generated by Notebook 03. It does not independently repeat raw-data cleaning, ensuring that final evaluation uses exactly the same cleaned population as feature engineering and model training.


In [ ]:
DATA_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "hotel_bookings_cleaned.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Processed dataset not found. Run Notebook 03 first."
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Cleaned shape:", df.shape)

assert df.shape == (119209, 30)


## 2. Apply the Shared Booking-Time Modelling Policy

The canonical transformation from `src/project_utils.py` applies the same booking-time exclusions and feature engineering used by Notebooks 05 and 06.

This prevents evaluation-time preprocessing from drifting away from the training definition.


In [ ]:
data = prepare_model_data(df)

forbidden = [
    "reservation_status",
    "reservation_status_date",
    *BOOKING_TIME_EXCLUSIONS,
]

remaining_forbidden = [
    c for c in forbidden
    if c in data.columns
]

print(
    "Booking-time exclusions:",
    BOOKING_TIME_EXCLUSIONS,
)
print(
    "Forbidden modelling columns remaining:",
    remaining_forbidden,
)
print("Model-ready shape:", data.shape)
print(
    "Cancellation rate:",
    round(data[TARGET].mean(), 4),
)

assert not remaining_forbidden
assert data.shape == (119209, 26)
assert int(data.isna().sum().sum()) == 0


## 3. Reproduce the Canonical Held-Out Split

The same deterministic 80/20 stratified split with `random_state=42` is reconstructed. The held-out partition has not been used for training-CV model comparison or hyperparameter selection.


In [ ]:
X = data.drop(columns=[TARGET])
y = data[TARGET].astype(int)

X_train, X_test, y_train, y_test = (
    stratified_train_test_split(X, y)
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print(
    "Train cancellation rate:",
    round(y_train.mean(), 4),
)
print(
    "Test cancellation rate:",
    round(y_test.mean(), 4),
)

assert X_train.shape == (95367, 25)
assert X_test.shape == (23842, 25)


## 4. Shared Preprocessing

The exact numerical, categorical, and rare-country preprocessing policy is imported from the shared project utilities. Rare-country grouping is learned from training data rather than from the held-out test set.


In [ ]:
(
    preprocessor,
    boost_preprocessor,
    num_cols,
    cat_cols,
) = build_preprocessors(X_train)

print(
    "Country policy: categories below 1% "
    "training frequency -> Other"
)
print("Numerical features:", len(num_cols))
print("Categorical features:", len(cat_cols))

assert len(num_cols) == 16
assert len(cat_cols) == 9


## 5. Reproduce Training-Only Decision Tree Selection

The Decision Tree tuning protocol from Notebook 06 is reproduced using training data only. The expected selected parameters are checked explicitly before held-out evaluation.


In [ ]:
tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ),
    ),
])

tree_grid = {
    "model__max_depth": [3, 5, 7, 10, 15, 20, None],
    "model__min_samples_leaf": [1, 5, 10],
}

tree_search = GridSearchCV(
    tree_pipeline,
    tree_grid,
    scoring="roc_auc",
    cv=CV,
    n_jobs=-1,
    refit=True,
)

tree_search.fit(X_train, y_train)

BEST_TREE_DEPTH = tree_search.best_params_[
    "model__max_depth"
]
BEST_TREE_MIN_LEAF = tree_search.best_params_[
    "model__min_samples_leaf"
]

print(
    "Best Decision Tree parameters:",
    tree_search.best_params_,
)
print(
    "Best training-CV ROC-AUC:",
    round(tree_search.best_score_, 4),
)

assert BEST_TREE_DEPTH == 15
assert BEST_TREE_MIN_LEAF == 10


## 6. Fit the Five Final Candidate Models

The five candidate families use the same definitions as Notebook 06. In particular, Random Forest uses the canonical 200-tree configuration and KNN remains the documented `k=5` baseline.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "KNN": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=5,
                n_jobs=-1,
            ),
        ),
    ]),

    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=BEST_TREE_DEPTH,
                min_samples_leaf=BEST_TREE_MIN_LEAF,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", make_random_forest()),
    ]),

    "HistGradientBoosting": Pipeline([
        ("preprocessor", boost_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                random_state=RANDOM_STATE
            ),
        ),
    ]),
}

for name, model in models.items():
    print("Fitting", name)
    model.fit(X_train, y_train)

rf_model = models["Random Forest"].named_steps["model"]

print("All final models fitted on training data.")
print(
    "Canonical Random Forest:",
    rf_model.n_estimators,
    "trees",
)

assert rf_model.n_estimators == 200


## 7. Single Held-Out Test Evaluation

After all model definitions and training-only choices are fixed, each model is evaluated on the held-out test set.

Accuracy, precision, recall, F1, ROC-AUC, and Average Precision are reported. The model ranking is interpreted primarily through threshold-independent discrimination metrics, while threshold-specific behaviour is examined separately.


In [ ]:
rows = []
predictions = {}
probabilities = {}

for name, model in models.items():
    prob = model.predict_proba(X_test)[:, 1]

    # Explicit probability decision rule so every reported
    # "threshold 0.50" result has identical semantics.
    pred = (prob >= 0.50).astype(int)

    predictions[name] = pred
    probabilities[name] = prob

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(
            y_test,
            pred,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_test,
            pred,
            zero_division=0,
        ),
        "F1": f1_score(
            y_test,
            pred,
            zero_division=0,
        ),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "Average_Precision":
            average_precision_score(y_test, prob),
    })

metrics_df = (
    pd.DataFrame(rows)
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

print("\nHeld-out test metrics — all five models:")
print(
    metrics_df
    .round(4)
    .to_string(index=False)
)

assert len(metrics_df) == 5
assert metrics_df.isna().sum().sum() == 0

best_model_name = metrics_df.iloc[0]["Model"]

print("\nBest held-out ROC-AUC model:", best_model_name)


### ROC-AUC vs precision–recall
ROC-AUC measures ranking across false-positive and true-positive rates and is useful for overall discrimination. **Precision–recall is particularly informative here because cancellation is the minority class (~37%) and it focuses directly on performance for the positive (cancelled) class.** We therefore report both rather than relying on accuracy alone.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)

    ax.plot(
        fpr,
        tpr,
        label=(
            f"{name} "
            f"(AUC={roc_auc_score(y_test, prob):.3f})"
        ),
    )

ax.plot([0, 1], [0, 1], "--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Five Models")
ax.legend(fontsize=8)

fig.tight_layout()

roc_path = FIGURES_DIR / "roc_curves.png"
fig.savefig(roc_path, dpi=180, bbox_inches="tight")

print("Saved:", roc_path)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, prob in probabilities.items():
    precision, recall, _ = precision_recall_curve(
        y_test,
        prob,
    )

    ap = average_precision_score(y_test, prob)

    ax.plot(
        recall,
        precision,
        label=f"{name} (AP={ap:.3f})",
    )

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(
    "Precision–Recall Curves — All Five Models"
)
ax.legend(fontsize=8)

fig.tight_layout()

pr_path = FIGURES_DIR / "precision_recall_curves.png"
fig.savefig(pr_path, dpi=180, bbox_inches="tight")

print("Saved:", pr_path)
plt.show()


## 8. Confusion Matrices at the Default 0.50 Threshold

Confusion matrices provide the concrete false-positive/false-negative trade-off at the conventional 0.50 threshold. Their counts are also saved as a machine-readable CSV for reproducibility.


In [ ]:
def safe_name(name):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        name.lower(),
    ).strip("_")


confusion_rows = []

for name in models:
    cm = confusion_matrix(
        y_test,
        predictions[name],
    )

    tn, fp, fn, tp = cm.ravel()

    confusion_rows.append({
        "Model": name,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    })

    fig, ax = plt.subplots(figsize=(6, 5))

    ConfusionMatrixDisplay(
        cm,
        display_labels=[
            "Not Cancelled",
            "Cancelled",
        ],
    ).plot(
        values_format="d",
        ax=ax,
    )

    ax.set_title(
        f"{name} — threshold 0.50"
    )

    fig.tight_layout()

    cm_path = (
        FIGURES_DIR
        / f"confusion_{safe_name(name)}.png"
    )

    fig.savefig(
        cm_path,
        dpi=180,
        bbox_inches="tight",
    )

    print("Saved:", cm_path)
    plt.show()

confusion_df = pd.DataFrame(confusion_rows)

print("\nHeld-out confusion counts at threshold 0.50:")
print(confusion_df.to_string(index=False))


## 9. Model Interpretation

Random Forest feature importance and Logistic Regression coefficients provide complementary descriptive views of the fitted predictors.

These quantities describe predictive associations in the fitted models and are not interpreted as causal effects.


In [ ]:
rf = models["Random Forest"]
lr = models["Logistic Regression"]

feature_names = (
    rf.named_steps["preprocessor"]
    .get_feature_names_out()
)

rf_imp = pd.Series(
    rf.named_steps["model"].feature_importances_,
    index=feature_names,
).sort_values(ascending=False)

lr_feature_names = (
    lr.named_steps["preprocessor"]
    .get_feature_names_out()
)

lr_coef = pd.Series(
    lr.named_steps["model"].coef_[0],
    index=lr_feature_names,
)

print("Top 15 Random Forest features")
print(
    rf_imp
    .head(15)
    .to_frame("RF_importance")
    .round(4)
    .to_string()
)

print(
    "\nTop 15 Logistic features "
    "by absolute coefficient"
)

lr_top = lr_coef.reindex(
    lr_coef
    .abs()
    .sort_values(ascending=False)
    .head(15)
    .index
)

print(
    lr_top
    .to_frame("Logistic_coefficient")
    .round(4)
    .to_string()
)


In [ ]:
top_features = rf_imp.head(15).index

comparison = pd.DataFrame({
    "RF_importance":
        rf_imp.reindex(top_features),

    "Logistic_coefficient":
        lr_coef.reindex(top_features),

    "Logistic_abs_coefficient":
        lr_coef.abs().reindex(top_features),
})

print(
    "\nRandom Forest importance "
    "vs Logistic coefficients:"
)

print(
    comparison
    .round(4)
    .to_string()
)

fig, ax = plt.subplots(figsize=(9, 6))

rf_imp.head(15).sort_values().plot(
    kind="barh",
    ax=ax,
)

ax.set_title(
    "Random Forest — Top 15 Feature Importances"
)
ax.set_xlabel("Importance")

fig.tight_layout()

importance_path = (
    FIGURES_DIR
    / "random_forest_feature_importance.png"
)

fig.savefig(
    importance_path,
    dpi=180,
    bbox_inches="tight",
)

print("Saved:", importance_path)
plt.show()


## 10. Persist Final Evaluation Artifacts

Final metrics, confusion counts, interpretation values, ROC/PR curves, confusion matrices, and the Random Forest importance plot are saved under `results/` so the repository preserves the evidence produced by the evaluation rather than relying only on transient notebook displays.


In [ ]:
metrics_path = RESULTS_DIR / "metrics.csv"
confusion_path = RESULTS_DIR / "confusion_counts.csv"
interpretation_path = (
    RESULTS_DIR
    / "rf_logistic_feature_comparison.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False,
)

confusion_df.to_csv(
    confusion_path,
    index=False,
)

comparison.to_csv(
    interpretation_path,
)

print("Saved final metrics:", metrics_path)
print("Saved confusion counts:", confusion_path)
print(
    "Saved interpretation comparison:",
    interpretation_path,
)

required_figures = [
    FIGURES_DIR / "roc_curves.png",
    FIGURES_DIR / "precision_recall_curves.png",
    FIGURES_DIR / "random_forest_feature_importance.png",
]

assert all(
    path.exists()
    for path in required_figures
)

print(
    "PASS: required final-evaluation "
    "artifacts were written."
)


## 11. Handoff to Notebook 08
Notebook 08 should use the final chosen model’s cancellation probabilities to define the false-positive/false-negative cost model, sweep probability thresholds, recommend **one threshold**, and report the confusion matrix at that recommended threshold.

**Important:** model quality and business threshold are separate decisions. A strong ranking model does not imply that 0.50 is the correct operational threshold.